# Meta DP ++ Algorithm Tomson Sampling (MTS++) based agents

> Agents utelizing the Meta DP TS ++ based approach for Dynamic pricing and learning problems from https://pubsonline.informs.org/doi/10.1287/mnsc.2021.4071

In [ ]:
#| default_exp agents.dynamic_pricing.PTS

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import logging

from abc import ABC, abstractmethod
from typing import Union, Optional, List
import numpy as np
import joblib
import os
import statsmodels.api as sm
from ddopai.agents.dynamic_pricing.utils import GLMLink
from ddopai.envs.base import BaseEnvironment
from ddopai.agents.dynamic_pricing.mushroom_rl import PricingMushroomBaseAgent
from mushroom_rl.core import Agent
from ddopai.utils import MDPInfo
from ddopai.agents.obsprocessors import FlattenTimeDimNumpy
from ddopai.envs.actionprocessors import ClipAction


In [ ]:
#| export
class _GaussianPosterior:
    """Stores (μ, Σ) and supports sampling and rank‑1 Bayesian updates."""
    def __init__(self, d: int, mu0: np.ndarray, Sigma0: np.ndarray, sigma: float, rng: np.random.Generator):
        self.d = d                      # dimension
        self.mu = mu0.astype(float).reshape(-1)
        self.Sigma = Sigma0.astype(float)
        self.sigma2 = sigma ** 2        # observation noise variance
        self.rng = rng
        # precision‑scaled mean f = Σ⁻¹ μ
        self.f = np.linalg.solve(self.Sigma, self.mu)

    # ---- API ----
    def sample_theta(self) -> np.ndarray:
        return self.rng.multivariate_normal(self.mu, self.Sigma)

    def update(self, m: np.ndarray, y: float):
        """Rank‑1 Bayesian update for y = θᵗ m + ε,  ε ~ N(0, σ²)."""
        m = m.reshape(-1)
        denom = self.sigma2 + m @ self.Sigma @ m
        gain = self.Sigma @ m / denom

        self.mu = self.mu + gain * (y - m @ self.mu)
        self.Sigma = self.Sigma - np.outer(gain, m) @ self.Sigma

        # Numerical stabilization: symmetrize and correct if not PSD
        self.Sigma = 0.5 * (self.Sigma + self.Sigma.T)
        eigvals = np.linalg.eigvalsh(self.Sigma)
        if eigvals.min() <= 0:
            self.Sigma += (abs(eigvals.min()) + 1e-10) * np.eye(self.Sigma.shape[0])

        # For completeness: update the precision-scaled mean (optional)
        self.f = np.linalg.solve(self.Sigma, self.mu)


    # reset (used when we drop in a new prior at epoch start)
    def reset(self, mu: np.ndarray, Sigma: np.ndarray):
        self.mu = mu.astype(float).reshape(-1)
        self.Sigma = Sigma.astype(float)
        self.f = np.linalg.solve(self.Sigma, self.mu)


In [ ]:
#| export
class PTSPolicy:
    def __init__(
        self,
        environment_info,
        prior_mean: np.ndarray,
        prior_cov: np.ndarray,
        sigma: float,
        lambda_e: float,
        ex_prices: Optional[np.ndarray] = None,
        price_function=None,
        actionprocessors: Optional[List[object]] = None,
        seed: Optional[int] = None,
        g=None,
    ):
        self.env_info = environment_info
        self.T = int(environment_info.horizon)
        self.p_min = float(environment_info.action_space.low)
        self.p_max = float(environment_info.action_space.high)
        self.d = int(environment_info.observation_space['features'].shape[0])
        self.sigma = float(sigma)
        self.lambda_e = float(lambda_e)
        self.price_function = price_function
        self.rng = np.random.default_rng(seed)
        self.actionprocessors = actionprocessors or []

        self.t = 0
        self.X_buf = np.empty((0, 2 * self.d))
        self.Y_buf = np.empty((0, 1))

        self.mu = np.concatenate([np.ones(self.d) * prior_mean[0], np.ones(self.d) * prior_mean[1]])
        self.Sigma = np.eye(2 * self.d) * prior_cov
        self.posterior = _GaussianPosterior(2 * self.d, self.mu, self.Sigma, self.sigma, self.rng)

        self.ex_prices = np.asarray(ex_prices, dtype=float) if ex_prices is not None else np.asarray([0.0, self.p_max])
        self.V = np.zeros((2 * self.d, 2 * self.d))

    def draw_action(self, observation: dict) -> float:
        x = observation['features'].astype(float).reshape(self.d)
        if np.linalg.eigvalsh(self.V).min() < self.lambda_e:
            price = self.ex_prices[self.t % len(self.ex_prices)]
        else:
            theta_sample = self.posterior.sample_theta()
            alpha = theta_sample[:self.d]
            beta = theta_sample[self.d:]
            price = self.price_function(x, alpha, beta)

        for proc in self.actionprocessors:
            price = proc(price)
        return price

    def fit(self, X: np.ndarray, Y: float, action: float):
        self.t += 1
        m = np.concatenate([X, X * action]).astype(float)
        self.X_buf = np.vstack([self.X_buf, m])
        self.Y_buf = np.vstack([self.Y_buf, [Y]])
        self.V += np.outer(m, m)

       
        self.posterior.update(m, float(Y))

    def update_task(self, env):
        """No-op in Algorithm 1 since prior is fixed."""
        self.env_info = env.mdp_info
        self.t = 0
        self.X_buf = np.empty((0, 2 * self.d))
        self.Y_buf = np.empty((0, 1))
        self.V = np.zeros((2 * self.d, 2 * self.d))
        self.posterior.reset(self.mu, self.Sigma)

    def reset(self):
        pass


In [ ]:
#| export
class PTSCoreAgent(Agent):

    """
    Base class for TS agents.
    """

    def __init__(self,
                environment_info,
                prior_mean: np.ndarray,
                prior_cov: np.ndarray,
                sigma: float,
                lambda_e: float,
                ex_prices: Optional[np.ndarray] = None,
                price_function=None,
                actionprocessors: Optional[List[object]] = None,
                seed: Optional[int] = None,
                agent_name: str = "PTSPolicy",
                g=None,
                 ):
        
        policy = PTSPolicy(
            environment_info,
            prior_mean=prior_mean,
            prior_cov=prior_cov,
            sigma=sigma,
            lambda_e=lambda_e,
            ex_prices=ex_prices,
            price_function=price_function,
            actionprocessors=actionprocessors,
            seed=seed,
            g=g,
        )
        self.agent_name = agent_name
        super().__init__(environment_info, policy)
        
    def fit(self, dataset, **kwargs):
        X = dataset[0][0]["features"]
        Y = kwargs["demand"][0]
        action = dataset[0][1]
        self.policy.fit(X, Y, action)
        
    def update_task(self, env):
        self.policy.update_task(env)


In [ ]:
#| export
class PTSAgent(PricingMushroomBaseAgent):
    """
    Wrapper class for TSCoreAgent to interact with MushroomRL.
    """
    def __init__(self,
                 environment_info,
                prior_mean: np.ndarray,
                prior_cov: np.ndarray,
                sigma: float,
                lambda_e: float,
                ex_prices: Optional[np.ndarray] = None,
                price_function=None,
                actionprocessors: Optional[List[object]] = None,
                seed: Optional[int] = None,
                agent_name: str = "PTSPolicy",
                obsprocessors: Optional[List[object]] = None,
                g: Optional[GLMLink] = None
                 ):
        self.agent = PTSCoreAgent(
            environment_info=environment_info,
            prior_mean=prior_mean,
            prior_cov=prior_cov,
            sigma=sigma,
            lambda_e=lambda_e,
            ex_prices=ex_prices,
            price_function=price_function,
            actionprocessors=actionprocessors,
            seed=seed,
            agent_name=agent_name,
            g=g,
        )
        super().__init__(environment_info=environment_info, obsprocessors=obsprocessors, agent_name=agent_name)
        
    def update_task(self, env: object):
        """ Update the environment specific parameters of the agent """
        self.agent.update_task(env)
